# ============================================================
# BƯỚC 2: PREPROCESSING (TIỀN XỬ LÝ DỮ LIỆU)
# ============================================================

**Mục tiêu:** Biến đổi dữ liệu thành dạng số mà máy tính có thể học được.

**Các bước:**
1. **StandardScaler**: Chuẩn hóa cột số (Age, RestingBP, Cholesterol, FastingBS, MaxHR, Oldpeak)
   - Công thức: Z = (x - mean) / std
   - Kết quả: Giá trị có mean=0, std=1

2. **OneHotEncoder**: Mã hóa cột phân loại (Sex, ChestPainType, RestingECG, ExerciseAngina, ST_Slope)
   - Biến mỗi giá trị thành cột 0/1 riêng
   - drop='first': Bỏ cột đầu để tránh đa cộng tuyến

3. **ColumnTransformer**: Kết hợp cả 2 bước trên trong 1 pipeline

In [1]:
# ============================================================
# BƯỚC 2.1: IMPORT THƯ VIỆN
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("✅ Đã import các thư viện thành công!")

✅ Đã import các thư viện thành công!


In [2]:
# ============================================================
# BƯỚC 2.2: LOAD DỮ LIỆU ĐÃ LÀM SẠCH
# ============================================================
# Load file heart_cleaned.csv (đã xử lý giá trị 0 ở Bước 1 - EDA)
df = pd.read_csv('../data/heart_cleaned.csv')

print("=== DỮ LIỆU ĐÃ LÀM SẠCH ===")
print(f"Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
print(f"Target: 0={sum(df['HeartDisease']==0)}, 1={sum(df['HeartDisease']==1)}")
print(f"\n5 dòng đầu tiên:")
print(df.head())

=== DỮ LIỆU ĐÃ LÀM SẠCH ===
Kích thước: 918 dòng, 12 cột
Target: 0=410, 1=508

5 dòng đầu tiên:
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [3]:
# ============================================================
# BƯỚC 2.3: TÁCH FEATURES (X) VÀ TARGET (y)
# ============================================================
# X: Các đặc điểm đầu vào (11 cột)
# y: Mục tiêu cần dự đoán (HeartDisease)
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

print("=== TÁCH DỮ LIỆU ===")
print(f"X (features): {X.shape} - {X.columns.tolist()}")
print(f"y (target):   {y.shape}")
print(f"\nPhân bố target:")
print(y.value_counts())

=== TÁCH DỮ LIỆU ===
X (features): (918, 11) - ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope']
y (target):   (918,)

Phân bố target:
HeartDisease
1    508
0    410
Name: count, dtype: int64


In [4]:
# ============================================================
# BƯỚC 2.4: TẠO PREPROCESSOR (ColumnTransformer)
# ============================================================
# Định nghĩa các cột số - cần chuẩn hóa bằng StandardScaler
# StandardScaler: Z = (x - mean) / std → mean=0, std=1
numeric_features = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

# Định nghĩa các cột phân loại - cần one-hot encoding
# OneHotEncoder: tạo cột 0/1 cho mỗi giá trị
# drop='first': bỏ cột đầu để tránh đa cộng tuyến
categorical_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Tạo ColumnTransformer - kết hợp cả 2 bước
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),  # Công thức 1: Chuẩn hóa cột số
        ('cat', OneHotEncoder(drop='first'), categorical_features)  # Công thức 2: Mã hóa cột chữ
    ]
)

print("=== PREPROCESSOR ĐÃ TẠO ===")
print(f"1. StandardScaler cho {len(numeric_features)} cột số:")
for f in numeric_features:
    print(f"   - {f}")
print(f"\n2. OneHotEncoder cho {len(categorical_features)} cột phân loại:")
for f in categorical_features:
    print(f"   - {f}")
print(f"\n3. drop='first' → bỏ cột đầu tiên của mỗi nhóm")

=== PREPROCESSOR ĐÃ TẠO ===
1. StandardScaler cho 6 cột số:
   - Age
   - RestingBP
   - Cholesterol
   - FastingBS
   - MaxHR
   - Oldpeak

2. OneHotEncoder cho 5 cột phân loại:
   - Sex
   - ChestPainType
   - RestingECG
   - ExerciseAngina
   - ST_Slope

3. drop='first' → bỏ cột đầu tiên của mỗi nhóm


In [5]:
# ============================================================
# BƯỚC 2.5: FIT-TRANSFORM DỮ LIỆU
# ============================================================
# fit(): Học tham số từ dữ liệu
#   - StandardScaler: tính mean và std của từng cột số
#   - OneHotEncoder: tìm các giá trị unique trong từng cột
# transform(): Áp dụng biến đổi lên dữ liệu
#   - StandardScaler: Z = (x - mean) / std
#   - OneHotEncoder: tạo cột 0/1 cho mỗi giá trị
# fit_transform() = fit() + transform() trong 1 lần gọi

print("Đang thực hiện fit-transform...")
X_processed = preprocessor.fit_transform(X)
print(f"✅ Hoàn thành! Kích thước: {X_processed.shape[0]} dòng, {X_processed.shape[1]} cột")

Đang thực hiện fit-transform...
✅ Hoàn thành! Kích thước: 918 dòng, 15 cột


In [6]:
# ============================================================
# BƯỚC 2.6: XEM KẾT QUẢ SAU PREPROCESSING
# ============================================================
# Lấy tên các cột sau khi xử lý
feature_names = []
for name, transformer, columns in preprocessor.transformers_:
    if name == 'num':
        # Cột số: giữ nguyên tên
        feature_names.extend(columns)
    elif name == 'cat':
        # Cột phân loại: lấy tên từ OneHotEncoder
        cat_names = transformer.get_feature_names_out(columns)
        feature_names.extend(cat_names)

# Tạo DataFrame để dễ nhìn
df_processed = pd.DataFrame(np.asarray(X_processed), columns=feature_names)
df_processed['HeartDisease'] = y.values

print("=== KẾT QUẢ SAU PREPROCESSING ===")
print(f"Kích thước: {df_processed.shape[0]} dòng, {df_processed.shape[1]} cột")
print(f"\nDanh sách {len(feature_names)} cột features:")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

print(f"\n5 dòng đầu tiên của dữ liệu đã xử lý:")
print(df_processed.head())

=== KẾT QUẢ SAU PREPROCESSING ===
Kích thước: 918 dòng, 16 cột

Danh sách 15 cột features:
  [ 0] Age
  [ 1] RestingBP
  [ 2] Cholesterol
  [ 3] FastingBS
  [ 4] MaxHR
  [ 5] Oldpeak
  [ 6] Sex_M
  [ 7] ChestPainType_ATA
  [ 8] ChestPainType_NAP
  [ 9] ChestPainType_TA
  [10] RestingECG_Normal
  [11] RestingECG_ST
  [12] ExerciseAngina_Y
  [13] ST_Slope_Flat
  [14] ST_Slope_Up

5 dòng đầu tiên của dữ liệu đã xử lý:
        Age  RestingBP  Cholesterol  FastingBS     MaxHR   Oldpeak  Sex_M  \
0 -1.433140   0.415002     0.908395  -0.551341  1.382928 -0.832432    1.0   
1 -0.478484   1.527329    -1.099021  -0.551341  0.754157  0.105664    0.0   
2 -1.751359  -0.141161     0.797895  -0.551341 -1.525138 -0.832432    1.0   
3 -0.584556   0.303769    -0.472855  -0.551341 -1.132156  0.574711    0.0   
4  0.051881   0.971166    -0.822771  -0.551341 -0.581981 -0.832432    1.0   

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0                1.0                0.

In [7]:
# ============================================================
# BƯỚC 2.7: LƯU DỮ LIỆU ĐÃ XỬ LÝ
# ============================================================
# Lưu ra file CSV để các bạn khác trong team có thể dùng
output_path = '../data/heart_preprocessed.csv'
df_processed.to_csv(output_path, index=False)

print("=== LƯU DỮ LIỆU ===")
print(f"✅ Đã lưu vào: {output_path}")
print(f"✅ Kích thước: {df_processed.shape[0]} dòng, {df_processed.shape[1]} cột")
print(f"✅ 15 cột features + 1 cột HeartDisease = {df_processed.shape[1]} cột")
print(f"\n📌 Các bạn khác có thể đọc bằng:")
print(f"   import pandas as pd")
print(f"   df = pd.read_csv('{output_path}')")

=== LƯU DỮ LIỆU ===
✅ Đã lưu vào: ../data/heart_preprocessed.csv
✅ Kích thước: 918 dòng, 16 cột
✅ 15 cột features + 1 cột HeartDisease = 16 cột

📌 Các bạn khác có thể đọc bằng:
   import pandas as pd
   df = pd.read_csv('../data/heart_preprocessed.csv')


---
## 📊 TỔNG KẾT BƯỚC 2 - PREPROCESSING

### Dữ liệu đầu vào:
- **918 dòng**, **11 cột** (6 số + 5 phân loại)
- File: `data/heart_cleaned.csv`

### Các bước đã thực hiện:
| Bước | Mô tả | Chi tiết |
|------|-------|----------|
| 1 | Load dữ liệu | Đọc `heart_cleaned.csv` |
| 2 | Tách X, y | X: 11 features, y: HeartDisease |
| 3 | Tạo preprocessor | StandardScaler + OneHotEncoder |
| 4 | Fit-transform | Học và áp dụng biến đổi |
| 5 | Lưu kết quả | Xuất ra `heart_preprocessed.csv` |

### Dữ liệu đầu ra:
- **918 dòng**, **16 cột** (15 features + 1 target)
- File: `data/heart_preprocessed.csv`
- **Sẵn sàng cho các thuật toán Machine Learning!** 🚀

### Giải thích 15 cột features:
| Nhóm | Số cột | Tên cột |
|------|--------|---------|
| **Số (chuẩn hóa)** | 6 | Age, RestingBP, Cholesterol, FastingBS, MaxHR, Oldpeak |
| **Sex** | 1 | Sex_M (1=Nam, 0=Nữ) |
| **ChestPainType** | 3 | ATA, NAP, TA (bỏ ASY) |
| **RestingECG** | 2 | Normal, ST (bỏ LVH) |
| **ExerciseAngina** | 1 | ExerciseAngina_Y (1=Có, 0=Không) |
| **ST_Slope** | 2 | Flat, Up (bỏ Down) |

---